# 11 -- Batter-Runner Advancement Analysis (Contact Luck v0.9)

Models what happens AFTER the batter safely reaches at least first on a fair
outfield air ball: `P(batter_final_base = held_at_first / advanced_to_second /
advanced_to_third / inside_the_park_home_run / retired_while_advancing)`. See
README.md "Batter-runner advancement (Version 0.9)" for the full writeup,
including two real bugs caught during development (a verb-lookup dict bug, and a
review-preamble parsing bug) and a real feature-omission bug caught by comparing
against a simple empirical baseline. Version 0.7/0.8 remain frozen/unaffected.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

DATA_PATH = Path("../data/processed/cleaned_development_data_with_sprint_speed.parquet")
df = pd.read_parquet(DATA_PATH) if DATA_PATH.exists() else None
print(f"Loaded {len(df):,} rows" if df is not None else "Data not found -- run make join-sprint-speed first")

Loaded 494,173 rows


## 1. The target is not a Statcast column -- parsed from `des`

`events` only records the batter's HIT TYPE, not where they ended up. The target
is reconstructed by `mlb_luck_score.eligibility.parse_batter_advancement_des`:
extract the batter's own name from the leading clause of `des`, strip any review/
challenge preamble, then match four specific trailing-clause patterns. Never
force-labeled -- an unresolved play is excluded, not guessed.

In [2]:
from mlb_luck_score.eligibility import (
    add_advancement_eligibility,
    add_outfield_opportunity_eligibility,
    compute_eligibility,
    summarize_advancement_exclusions,
    summarize_advancement_parse_coverage,
)

prepared = None
if df is not None:
    prepared = compute_eligibility(df)
    prepared = add_outfield_opportunity_eligibility(prepared)
    prepared = add_advancement_eligibility(prepared)
    exclusions = summarize_advancement_exclusions(prepared)
    print(f"{prepared['advancement_eligible'].sum():,} eligible rows")
    print()
    print(pd.Series(exclusions.total).sort_values(ascending=False))
    print()
    coverage = summarize_advancement_parse_coverage(prepared)
    print(json.dumps(coverage, indent=2))

85,282 eligible rows

not_outfield_air_ball_bb_type         248500
not_advancement_safe_event            137952
eligible                               85282
trivial_no_advancement_opportunity     22436
unresolved_des_parse                       3
dtype: int64



{
  "__all_parsed_scope__": {
    "parsed_unambiguous": 107718,
    "unsupported_play_sequence": 3
  },
  "held_at_first": {
    "n_rows": 53417
  },
  "advanced_to_second": {
    "n_rows": 27982
  },
  "advanced_to_third": {
    "n_rows": 2910
  },
  "inside_the_park_home_run": {
    "n_rows": 86
  },
  "retired_while_advancing": {
    "n_rows": 887
  },
  "reached_on_error": {
    "n_rows": 1345
  },
  "multi_clause_or_multi_throw": {
    "n_rows": 643
  }
}


In [3]:
if prepared is not None:
    print(prepared['batter_final_base'].value_counts(dropna=False))

batter_final_base
None                        408891
held_at_first                53417
advanced_to_second           27982
advanced_to_third             2910
retired_while_advancing        887
inside_the_park_home_run        86
Name: count, dtype: int64


## 2. Feature engineering

`hit_type_group` (the batter's own hit type: single/double/triple) is a genuine,
non-leakage feature -- it establishes only the FLOOR of possible outcomes, not the
final result. Omitting it was a real bug caught in Section 4 below.

In [4]:
from mlb_luck_score.features.build_contact_features import add_advancement_features

if prepared is not None:
    prepared = add_advancement_features(prepared)
    eligible_preview = prepared[prepared["advancement_eligible"].astype(bool)]
    print(eligible_preview["hit_type_group"].value_counts())

hit_type_group
single_or_error    55643
double             27139
triple              2462
Name: count, dtype: int64


## 3. Fit the contact model (train seasons only) and model selection (fit 2021-2022, select on 2023)

In [5]:
from mlb_luck_score.models.compare_advancement_models import (
    fit_advancement_contact_model,
    get_advancement_rows,
    run_advancement_model_selection,
)

contact_trained = None
advancement_df = None
winner_variant = None
selection_metrics = None
trained_by_candidate = None
if prepared is not None:
    contact_trained = fit_advancement_contact_model(prepared)
    advancement_df = get_advancement_rows(prepared, contact_trained)
    winner_variant, selection_metrics, trained_by_candidate = run_advancement_model_selection(advancement_df)
    print("winner:", winner_variant)
    print(json.dumps(selection_metrics, indent=2))

winner: advancement_speed_v09
{
  "advancement_context_v09": {
    "multiclass_log_loss": 0.1548573012586105,
    "mean_one_vs_rest_ece": 0.0026789695978289376
  },
  "advancement_speed_v09": {
    "multiclass_log_loss": 0.15423102570836222,
    "mean_one_vs_rest_ece": 0.0023201679093087626
  },
  "advancement_nonlinear_v09_candidate": {
    "multiclass_log_loss": 0.27222302656442415,
    "mean_one_vs_rest_ece": 0.004679769688796398
  }
}


## 4. Final comparison against the empirical baseline (2024)

The empirical baseline (outcome frequency by hit-type-implied floor base x
sprint-speed quartile, fit ONLY on 2021-2022) exists specifically so the learned
candidates have something concrete to be checked against. This caught a real bug:
an earlier feature set omitting `hit_type_group` let ALL THREE learned candidates
score worse than this two-column baseline.

In [6]:
from mlb_luck_score.config import CALIBRATION_BASE_TRAIN_SEASONS
from mlb_luck_score.models.compare_advancement_models import (
    fit_empirical_advancement_baseline,
    run_advancement_final_comparison,
)

overall_comparison = None
final_df = None
winner_proba = None
empirical_baseline = None
if winner_variant is not None:
    winner_trained = trained_by_candidate[winner_variant]
    fit_df = advancement_df[advancement_df["season"].isin(CALIBRATION_BASE_TRAIN_SEASONS)]
    empirical_baseline = fit_empirical_advancement_baseline(fit_df)
    overall_comparison, final_df, winner_proba = run_advancement_final_comparison(
        advancement_df, winner_trained, empirical_baseline
    )
    print(json.dumps(overall_comparison, indent=2))

{
  "advancement_speed_v09": {
    "variant": "advancement_speed_v09",
    "sample_count": 21200,
    "multiclass_log_loss": 0.1553008615637386,
    "per_class_ece": {
      "held_at_first": 0.004504682060845955,
      "advanced_to_second": 0.004052974718542993,
      "advanced_to_third": 0.0014162530084474443,
      "inside_the_park_home_run": 0.0001592557355179661,
      "retired_while_advancing": 0.002886164216423252
    },
    "mean_expected_advancement_index": 1.3838812648801995,
    "class_frequencies": {
      "held_at_first": 13404,
      "advanced_to_second": 6812,
      "advanced_to_third": 743,
      "inside_the_park_home_run": 20,
      "retired_while_advancing": 221
    }
  },
  "advancement_empirical_baseline": {
    "variant": "advancement_empirical_baseline",
    "sample_count": 21200,
    "multiclass_log_loss": 0.18112300985216048,
    "per_class_ece": {
      "held_at_first": 0.0017031101427931877,
      "advanced_to_second": 0.004875349722886661,
      "advanced_to_t

## 5. Perturbation check

`sprint_speed_direction` is REQUIRED: a faster batter-runner must not show a lower
expected advancement index than a slower one, holding contact/context fixed.

In [7]:
from mlb_luck_score.models.compare_advancement_models import run_advancement_perturbation_checks

perturbation = None
if final_df is not None:
    perturbation = run_advancement_perturbation_checks(winner_trained, final_df)
    for name, r in perturbation.items():
        print(f"{name}: passed={r.passed} delta={r.delta:.4f} (low={r.mean_expected_advancement_low:.4f}, high={r.mean_expected_advancement_high:.4f}, n={r.sample_size})")

sprint_speed_direction: passed=True delta=0.0151 (low=1.3735, high=1.3886, n=21112)


## 6. Required-subgroup and venue calibration gate (v0.7D machinery, reused per one-vs-rest class)

Since there is no prior production advancement model, `baseline_p_out =
specialist_p_out` (self-baseline) -- the paired-delta CI is identically zero and
can never trigger a "credible regression" finding, leaving only the absolute-ECE
-confidence-interval criterion active, looped over every (subgroup, class) pair.

In [8]:
from mlb_luck_score.models.compare_advancement_models import evaluate_all_advancement_subgroups

subgroup_evidence = None
if final_df is not None:
    subgroup_evidence = evaluate_all_advancement_subgroups(final_df, winner_proba)
    evidence_df = pd.DataFrame(
        [{"label": e.label, "status": e.status, "n_plays": e.n_plays, "ece": e.specialist_adaptive_ece} for e in subgroup_evidence]
    )
    print(evidence_df["status"].value_counts())
    print()
    print(evidence_df[evidence_df["status"] == "not_calibrated"])

status
calibrated               141
insufficient_evidence    114
Name: count, dtype: int64

Empty DataFrame
Columns: [label, status, n_plays, ece]
Index: []


In [9]:
from mlb_luck_score.models.compare_advancement_models import summarize_advancement_calibration

gate_summary = None
if subgroup_evidence is not None:
    gate_summary = summarize_advancement_calibration(
        overall_comparison, winner_variant, subgroup_evidence, perturbation
    )
    print("beats_empirical_baseline:", gate_summary["beats_empirical_baseline"])
    print("overall_status:", gate_summary["overall_status"])
    print("calibrated_advancement_model:", gate_summary["calibrated_advancement_model"])
    print("calibrated:", len(gate_summary["calibrated_pairs"]))
    print("not_calibrated:", gate_summary["not_calibrated_pairs"])
    print("insufficient_evidence count:", len(gate_summary["insufficient_evidence_pairs"]))

beats_empirical_baseline: True
overall_status: calibrated_with_limited_subgroup_evidence
calibrated_advancement_model: False
calibrated: 141
not_calibrated: []
insufficient_evidence count: 114


## 7. Six-component report (never summed)

```
expected_contact_base           -- existing contact model's own pre-fielding prediction
actual_final_batter_base        -- the real observed outcome
advancement_opportunity         -- this model's predicted expected base value
batter_runner_advancement_execution = actual - opportunity
defensive_advancement_effect        = -batter_runner_advancement_execution
residual_uncertainty            -- variance of the predicted base-value distribution
```

In [10]:
from mlb_luck_score.config import CALIBRATION_EVAL_SEASONS
from mlb_luck_score.features.build_contact_features import add_advancement_contact_probability_features
from mlb_luck_score.models.train_contact_model import predict_proba_ordered
from mlb_luck_score.scoring.advancement_execution import build_advancement_component_report

report = None
if gate_summary is not None:
    air_ball_2024 = prepared[
        prepared["bb_type"].isin(["fly_ball", "line_drive"]) & prepared["season"].isin(CALIBRATION_EVAL_SEASONS)
    ]
    contact_feature_cols = contact_trained.numeric_features + contact_trained.categorical_features
    contact_proba_full = predict_proba_ordered(contact_trained, air_ball_2024[contact_feature_cols])
    air_ball_2024 = add_advancement_contact_probability_features(air_ball_2024, contact_proba_full)

    report = build_advancement_component_report(
        air_ball_2024, contact_trained, winner_trained, overall_status=gate_summary["overall_status"]
    )
    print(report["advancement_status"].value_counts())
    print()
    print(report[report["advancement_status"] == "provisional_advancement_model"].head(10))

advancement_status
unavailable_missing_inputs       35552
provisional_advancement_model    21200
excluded_trivial_home_run         5439
Name: count, dtype: int64

        game_pk  at_bat_number  pitch_number  expected_contact_base  actual_final_batter_base  advancement_opportunity  batter_runner_advancement_execution  \
370199   744795             17             1               1.060317                       1.0                 1.001644                            -0.001644   
370201   744795             19             2               0.637051                       1.0                 1.006492                            -0.006492   
370206   744795             25             1               0.651245                       1.0                 1.012999                            -0.012999   
370221   744795             44             1               0.726004                       1.0                 1.055279                            -0.055279   
370230   744795             59            

## 8. Explicit limitations

- `calibrated_advancement_model` is `False` pending more data specifically for
  `inside_the_park_home_run` (n=86) and `retired_while_advancing` (n=887) --
  zero (subgroup, class) pairs are credibly `not_calibrated`.
- Infield hits, preexisting-runner advancement, and discretionary scorer decisions
  are explicitly out of scope for this phase.
- `defensive_advancement_effect` is a sign-flipped restatement of the SAME
  execution quantity, not an independent estimate of the defense's own
  contribution -- a single joint model cannot cleanly separate batter-speed
  effects from defensive-positioning effects.
- Plain over-the-fence home runs are excluded by design -- no genuine advancement
  opportunity exists once a ball clears the fence.
- Version 0.7 (outfield opportunity/execution) and Version 0.8 (infield
  opportunity/execution) remain frozen and untouched by this notebook.